In [ ]:
import pandas as pd
import numpy as np
import xarray as xr
import os
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
lake = 'neuchatel'
base_folder = rf"/storage/alplakes_test/{lake}_100m_2025"
ke_folder = os.path.join(base_folder, "outputs_swirl_params06", "ke_eddy")
wind_folder = os.path.join(base_folder, 'wind_analysis')

# Prepare data

### Import datasets

In [ ]:
ds_wind_RW = xr.open_dataarray(os.path.join(wind_folder, 'mean_wind_RW_Wperm2.nc')) # W/m2
E_wind_MJperh = xr.open_dataarray(os.path.join(wind_folder, 'E_wind_MJperh.nc')) # MJ/h

In [ ]:
ds_wind_RW['time'] = pd.to_datetime(ds_wind_RW['time']) # W/m2
E_wind_MJperh['time'] = pd.to_datetime(E_wind_MJperh['time']) # MJ/h

In [ ]:
lake_ke = pd.read_csv(os.path.join(base_folder, "energy_budget", "ke_whole_lake.csv"), index_col=1).drop(columns=["Unnamed: 0"])
eddy_ke = pd.read_csv(os.path.join(ke_folder, "ke_eddies.csv"), index_col=1).drop(columns=["Unnamed: 0"])
lake_upper_layer_ke = lake_ke #pd.read_csv(os.path.join(ke_folder, "ke_lake.csv"), index_col=1).drop(columns=["Unnamed: 0"])

In [ ]:
lake_ke.index = pd.to_datetime(lake_ke.index)
eddy_ke.index = pd.to_datetime(eddy_ke.index)
lake_upper_layer_ke.index = pd.to_datetime(lake_upper_layer_ke.index)

In [ ]:
(eddy_ke['kinetic_energy_eddy_[MJ]']/lake_ke['kinetic_energy_[MJ]']).mean()

In [ ]:
eddy_ke = eddy_ke[eddy_ke.index.isin(lake_ke.index)]

In [ ]:
plt.scatter(eddy_ke['kinetic_energy_eddy_[MJ]'], lake_ke['kinetic_energy_[MJ]'])

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))
ax1.plot(pd.to_datetime(lake_ke.index), lake_ke['kinetic_energy_[MJ]'], label='Lake Kinetic Energy')
ax2=ax1.twinx()
ax2.plot(pd.to_datetime(eddy_ke.index), eddy_ke['kinetic_energy_eddy_[MJ]'],color='red', label='Eddy Kinetic Energy')
ax2.legend(fontsize=11, loc='upper left')
ax1.legend(fontsize=11, loc='upper right')
ax1.set_ylabel('Energy (MJ)')
ax1.set_title('Wind Energy Input vs. Lake and Eddy Kinetic Energy')
plt.tight_layout()
plt.show()


In [ ]:
E_wind_MJperh.plot()

### Crop to start date

In [ ]:
crop_date = '2025-01-01'
ds_wind_RW_crop = ds_wind_RW.sel(time=ds_wind_RW.time >= np.datetime64(crop_date))
E_wind_MJperh_crop = E_wind_MJperh.sel(time=E_wind_MJperh.time >= np.datetime64(crop_date))
eddy_ke = eddy_ke.loc[eddy_ke.index >= crop_date]
lake_ke = lake_ke.loc[lake_ke.index >= crop_date]
lake_upper_layer_ke = lake_upper_layer_ke.loc[lake_upper_layer_ke.index >= crop_date]

In [ ]:
lake_upper_layer_ke

### Compute dissipation

In [ ]:
dKE_dt = lake_ke['kinetic_energy_[MJ]'].diff() #MJ/h
dKE_up_dt = lake_upper_layer_ke['kinetic_energy_[MJ]'].diff() #MJ/h

#dissipation_MJ_per_h = E_wind_MJperh.values - dKE_dt.values
dissipation_MJ_per_h = -1* dKE_dt.values
dissipation_MJ_per_h[dissipation_MJ_per_h < 0] = 0

# Dissipation rate: epsilon = E_wind_input - dKE/dt  [MJ/h]
dissipation = pd.DataFrame({
    'E_wind_input_[MJ/h]': E_wind_MJperh.values,
    'dKE_dt_[MJ/h]': dKE_dt.values,
    'dKE_up_dt_[MJ/h]': dKE_up_dt.values,
    'dissipation_[MJ/h]': dissipation_MJ_per_h
}, index=lake_ke.index)
dissipation.index.name = 'date'

In [ ]:
E_wind_MJperh.time

### Rolling average

In [ ]:
rolling_window = 1

In [ ]:
# Rolling average
dissipation_rolling = dissipation.rolling(window=rolling_window, center=True, min_periods=1).mean()
lake_ke_rolling = lake_ke[['ke_mj_total']].rolling(window=rolling_window, center=True, min_periods=1).mean()
lake_upper_layer_ke_roll = lake_upper_layer_ke[['kinetic_energy_[MJ]']].rolling(window=rolling_window, center=True, min_periods=1).mean()

E_wind_MJ_rolling = E_wind_MJperh.rolling(time=rolling_window, min_periods=1).sum()

# Plot timeseries

In [ ]:
plt.figure(figsize=[20,8])
dissipation_rolling['dKE_dt_[MJ/h]'].plot()
dissipation_rolling['E_wind_input_[MJ/h]'].plot()
dissipation_rolling['dissipation_[MJ/h]'].plot()
plt.legend()

# Dissipation statistics

In [ ]:
diss = dissipation['dissipation_[MJ/h]'].iloc[1:]
diss = diss[diss>0]

In [ ]:
diss

In [ ]:
diss_in_W = diss * 1e6 / 3600 # MJ/h --> J/s

In [ ]:
# volume_lake = 6896096543.084367 # m3 --> 1st 70m of the lake
volume_lake = 12233766912.0 # m3 --> entire lake
rho_w = 1000

In [ ]:
diss_in_Wperkg = diss_in_W / (rho_w * volume_lake) # W/kg

In [ ]:
ts = diss_in_Wperkg
unit = 'W/kg'

# Basic statistics
print("Descriptive statistics:")
print(ts.describe())

# Plot time series
plt.figure(figsize=(12, 6))
plt.plot(ts, label=f'Dissipation [{unit}]')
plt.title('Time Series of Dissipation')
plt.xlabel('Time index')
plt.ylabel(f'Dissipation [{unit}]')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Histogram of values
plt.figure(figsize=(8, 5))
sns.histplot(ts, bins=30, kde=True, color='skyblue')
plt.title('Histogram of Dissipation Values')
plt.xlabel(f'Dissipation [{unit}]')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Boxplot to visualize spread and outliers
plt.figure(figsize=(6, 4))
sns.boxplot(x=ts, color='lightgreen')
plt.xscale('log')   # <-- log scale here
plt.title('Boxplot of Dissipation')
plt.xlabel(f'Dissipation [{unit}]')
plt.show()

In [ ]:
ts_log = np.log10(ts)

# Violin plot
plt.figure(figsize=(6, 8))
sns.violinplot(y=ts_log, inner='quartile', color='skyblue')  # inner='quartile' shows median & quartiles
plt.title('Violin Plot of Dissipation [W/kg] (log10)')
plt.ylabel('log10(Dissipation) [W/kg]')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()


In [ ]:
ts.name = 'diss_in_Wperkg'
ts.to_csv(os.path.join(ke_folder, "dissipation_without_wind.csv"))

In [ ]:
lake_ke_in_Jperkg = 1e6 * lake_ke/(volume_lake*rho_w)
lake_ke_in_Jperkg = lake_ke_in_Jperkg.rename(columns={'ke_mj_total': 'ke_in_Jperkg'})

In [ ]:
lake_ke_in_Jperkg.to_csv(os.path.join(ke_folder, "ke_in_Jperkg.csv"))

In [ ]:
lake_ke_in_Jperkg_log = np.log10(lake_ke_in_Jperkg)['ke_mj_total']

# Violin plot
plt.figure(figsize=(6, 8))
sns.violinplot(y=lake_ke_in_Jperkg_log, inner='quartile', color='skyblue')  # inner='quartile' shows median & quartiles
plt.title('Violin Plot of Kinetic Energy density [J/kg] (log10)')
plt.ylabel('log10(Kinetic Energy density) [J/kg]')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

# Figures whole lake

In [ ]:
def get_linear_trend_curve(x_data, y_data):
    valid = np.isfinite(x_data) & np.isfinite(y_data)
    x = x_data[valid].values
    y = y_data[valid].values

    # --- Linear fit ---
    m, b = np.polyfit(x, y, 1)

    # Trend line
    x_sorted = np.linspace(x.min(), x.max(), 300)
    y_trend = m * x_sorted + b

    # Predictions at original x (for fit metrics)
    y_hat = m * x + b
    resid = y - y_hat

    # --- Fit quality ---
    ss_res = np.sum(resid**2)
    ss_tot = np.sum((y - np.mean(y))**2)
    r2 = 1 - ss_res / ss_tot

    rmse = np.sqrt(np.mean(resid**2))
    mae = np.mean(np.abs(resid))

    return (x_sorted, y_trend), r2, rmse, mae

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
E_wind_MJ_rolling.plot(ax=ax, label=f'Wind Energy Input (rolling {rolling_window}h sum)')
ax.plot(pd.to_datetime(lake_ke.index), lake_ke['ke_mj_total'], label='Lake Kinetic Energy')
ax.plot(pd.to_datetime(lake_upper_layer_ke.index), lake_upper_layer_ke['kinetic_energy_[MJ]'], label='Lake Kinetic Energy - Upper Layer')
ax.plot(pd.to_datetime(eddy_ke.index), eddy_ke['kinetic_energy_eddy_[MJ]'], label='Eddy Kinetic Energy')
ax.legend(fontsize=11, loc='upper right')
ax.set_ylabel('Energy (MJ)')
ax.set_title('Wind Energy Input vs. Lake and Eddy Kinetic Energy')
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Top panel: energy budget components
axes[0].plot(dissipation_rolling.index, dissipation_rolling['E_wind_input_[MJ/h]'], label=f'Wind Energy Input ({rolling_window}h avg)',
             alpha=0.8)
axes[0].plot(dissipation_rolling.index, dissipation_rolling['dKE_dt_[MJ/h]'], label=f'dKE/dt ({rolling_window}h avg)', alpha=0.8)
axes[0].set_ylabel('Energy rate (MJ/h)')
axes[0].set_title(f'Energy Budget Components – {rolling_window}h Rolling Average')
axes[0].axhline(0, color='gray', linewidth=0.5, linestyle='--')

# Right axis for total KE on top panel
ax0_right = axes[0].twinx()
ax0_right.plot(lake_ke_rolling.index, lake_ke_rolling['kinetic_energy_[MJ]'], label=f'Total KE ({rolling_window}h avg)', color='C2',
               alpha=0.7, linestyle='--')
ax0_right.set_ylabel('Total KE (MJ)')

# Combine legends from both axes
lines0, labels0 = axes[0].get_legend_handles_labels()
lines0r, labels0r = ax0_right.get_legend_handles_labels()
axes[0].legend(lines0 + lines0r, labels0 + labels0r, fontsize=11)

# Bottom panel: dissipation rate
axes[1].plot(dissipation_rolling.index, dissipation_rolling['dissipation_[MJ/h]'], label=f'Dissipation ({rolling_window}h avg)',
             color='C3',
             alpha=0.8)
axes[1].set_ylabel('Dissipation (MJ/h)')
axes[1].set_xlabel('Date')
axes[1].set_title(f'Dissipation : ε = E_wind - dKE/dt – {rolling_window}h Rolling Average')
axes[1].axhline(0, color='gray', linewidth=0.5, linestyle='--')

# Right axis for total KE on bottom panel
ax1_right = axes[1].twinx()
ax1_right.plot(lake_ke_rolling.index, lake_ke_rolling['kinetic_energy_[MJ]'], label=f'Total KE ({rolling_window}h avg)', color='C2',
               alpha=0.7, linestyle='--')
ax1_right.set_ylabel('Total KE (MJ)')

# Combine legends from both axes
lines1, labels1 = axes[1].get_legend_handles_labels()
lines1r, labels1r = ax1_right.get_legend_handles_labels()
axes[1].legend(lines1 + lines1r, labels1 + labels1r, fontsize=11)

plt.tight_layout()
plt.show()


In [ ]:
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
from bokeh.models import HoverTool
from bokeh.models import LinearAxis, Range1d

output_notebook()

In [ ]:
from bokeh.plotting import figure, show
from bokeh.models import (
    ColumnDataSource, HoverTool,
    Range1d, LinearAxis
)

# --- Prepare data sources ---
src = ColumnDataSource(data=dict(
    date=dissipation_rolling.index,
    dissipation=dissipation_rolling['dissipation_[MJ/h]'].values,
    ke=lake_upper_layer_ke['kinetic_energy_[MJ]'].values
))

# --- Create the base figure (left axis) ---
p = figure(
    title=f'Dissipation Rate (ε = E_wind - dKE/dt) – {rolling_window}h Rolling Average',
    x_axis_type='datetime',
    width=1200,
    height=450,
    x_axis_label='Date',
    y_axis_label='Dissipation (MJ/h)',
    tools='xpan,xwheel_zoom,box_zoom,reset,save',
    active_scroll='xwheel_zoom',
)

# Determine the ranges
left_min = dissipation_rolling['dissipation_[MJ/h]'].min()
left_max = dissipation_rolling['dissipation_[MJ/h]'].max()

right_min = lake_upper_layer_ke['kinetic_energy_[MJ]'].min()
right_max = lake_upper_layer_ke['kinetic_energy_[MJ]'].max()

# Set left y-range explicitly
p.y_range = Range1d(start=left_min, end=left_max)

# Add right y-axis
p.extra_y_ranges = {"ke_range": Range1d(start=right_min, end=right_max)}
p.add_layout(
    LinearAxis(y_range_name="ke_range", axis_label="KE upper layer (MJ)"),
    'right'
)

# Plot left axis line
p.line(
    x='date', y='dissipation',
    source=src,
    legend_label='Dissipation Rate',
    line_color='#E45756',
    line_width=1.5,
    alpha=0.8
)

# Plot right axis line
p.line(
    x='date', y='ke',
    source=src,
    legend_label='KE upper layer',
    line_width=1.5,
    alpha=0.8,
    y_range_name="ke_range"
)

# Hover (now works properly)
p.add_tools(HoverTool(
    tooltips=[
        ("Date", "@date{%F %H:%M}"),
        ("Dissipation (MJ/h)", "@dissipation{0.00}"),
        ("KE upper layer (MJ)", "@ke{0.00}")
    ],
    formatters={"@date": "datetime"},
    mode="vline"
))

p.legend.location = 'top_right'
p.legend.click_policy = 'hide'

show(p)


In [ ]:
# Filter to only positive dissipation values
positive_mask = dissipation_rolling['dissipation_[MJ/h]'].iloc[1:] > 0

x_data = lake_ke_rolling['kinetic_energy_[MJ]'].iloc[1:][positive_mask]
y_data = dissipation_rolling['dissipation_[MJ/h]'].iloc[1:][positive_mask]

# Fit a linear trend curve
(x_trend, y_trend), r2, rmse, mae = get_linear_trend_curve(x_data, y_data)

fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(
    x_data,
    y_data,
    s=5, alpha=0.3, label='Data'
)
ax.plot(x_trend, y_trend, color='red', linewidth=2, label=f'Trend RMSE={rmse:.3f}, R2={r2:.3f}')
ax.set_xlabel(f'Total Lake KE – {rolling_window}h Rolling Avg (MJ)')
ax.set_ylabel(f'Dissipation Rate – {rolling_window}h Rolling Avg (MJ/h)')
ax.set_title(f'Dissipation Rate vs. Total Lake Kinetic Energy ({rolling_window}h Rolling Average) – Positive Dissipation Only')
ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Filter to only positive dissipation values
positive_mask = dissipation_rolling['dissipation_[MJ/h]'].iloc[1:] > 0

x_data = lake_upper_layer_ke['kinetic_energy_[MJ]'].shift(1).iloc[1:][positive_mask]
y_data = dissipation_rolling['dissipation_[MJ/h]'].iloc[1:][positive_mask]

# Fit a linear trend curve
(x_trend, y_trend), r2, rmse, mae = get_linear_trend_curve(x_data, y_data)

fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(
    x_data,
    y_data,
    s=5, alpha=0.3, label='Data'
)
ax.plot(x_trend, y_trend, color='red', linewidth=2, label=f'Trend RMSE={rmse:.3f}, R2={r2:.3f}')
ax.set_xlabel(f'KE in Lake Upper Layer – {rolling_window}h Rolling Avg (MJ)')
ax.set_ylabel(f'Dissipation Rate – {rolling_window}h Rolling Avg (MJ/h)')
ax.set_title(f'Dissipation Rate vs. Lake Kinetic Energy in Upper Layer ({rolling_window}h Rolling Average) – Positive Dissipation Only')
ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
E_wind_MJ_24h_sum = E_wind_MJperh.rolling(time=rolling_window, min_periods=1).sum()
E_wind_24h_sum_aligned = E_wind_MJ_24h_sum.sel(time=lake_ke.index, method='nearest')

positive_mask = dissipation_rolling['dissipation_[MJ/h]'].iloc[1:] > 0

x_neg = pd.Series(E_wind_24h_sum_aligned.values[1:], index=dissipation_rolling.index[1:])[positive_mask]
y_neg = dissipation_rolling['dissipation_[MJ/h]'].iloc[1:][positive_mask]

# Fit a linear trend curve
(x_trend, y_trend), r2, rmse, mae = get_linear_trend_curve(x_neg, y_neg)

fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(x_neg, y_neg, s=5, alpha=0.3, label='Data')
ax.plot(x_trend, y_trend, color='red', linewidth=2, label=f'Trend RMSE={rmse:.3f}, R2={r2:.3f}')
ax.set_xlabel(f'Wind Energy Input – {rolling_window}h Rolling Total (MJ)')
ax.set_ylabel(f'Dissipation Rate – {rolling_window}h Rolling Avg (MJ/h)')
ax.set_title(f'Positive Dissipation Rate vs. Wind Energy Input (total past {rolling_window}h)')
ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
x_data = lake_ke_rolling['kinetic_energy_[MJ]'].iloc[1:]
y_data = dissipation_rolling['dissipation_[MJ/h]'].iloc[1:]

fig, ax1 = plt.subplots(figsize=(15, 5))
i_start = 24 * 18
ax1.plot(y_data[i_start:i_start + 24 * 10], label='Dissipation Rate')
ax2 = ax1.twinx()
ax2.plot(x_data[i_start:i_start + 24 * 10], color='C1', label='Total KE')

# Add wind energy on the left axis
E_wind_rolling_aligned = dissipation_rolling['E_wind_input_[MJ/h]'].iloc[1:]
ax1.plot(E_wind_rolling_aligned[i_start:i_start + 24 * 10], color='C2', label='Wind Energy Input')

# Add dKE/dt on the left axis
dKE_dt_rolling_aligned = -1*dissipation_rolling['dKE_dt_[MJ/h]'].iloc[1:]
ax1.plot(dKE_dt_rolling_aligned[i_start:i_start + 24 * 10], color='C3', label='-dKE/dt')

# Add title with rolling window
ax1.set_title(f'Energy Budget Components – {rolling_window}h Rolling Window')

# Add y-axis labels
ax1.set_ylabel('Energy Rate (MJ/h)')
ax2.set_ylabel('Total KE (MJ)')

# Combine legends from both axes
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2)


# Check energy spectrum to see if we can easily remove IW influence

In [ ]:
from utils_signal_processing import *

In [ ]:
xr_diss = dissipation_rolling.to_xarray()

In [ ]:
ke_fft = xr_compute_meanfft(xr_diss['dissipation_[MJ/h]'].rename({'date':'time'}), M=1)

In [ ]:
cutoff1_hr = 138
cutoff2_hr = 3

cutoff1 = 1/(cutoff1_hr * 3600)
cutoff2 = 1/(cutoff2_hr * 3600)

In [ ]:
fig,ax = plot_freq_spectrum(ke_fft, 'KE', depth=0, m_segm=1, y_lim_min=1e-15, x_lim_min=1e-8, fontsize=10)
ax.axvline(x = cutoff1, linestyle="--", color="k",label="cutoff1")
ax.axvline(x = cutoff2, linestyle="--", color="k",label="cutoff2")
ax.legend()